# Technology — Apache Airflow: Pipeline Orchestration
---

<div style="padding:15px;border-left:8px solid #1a8a5a;background:#e8f5e9;border-radius:4px;">
<strong>Core Insight:</strong> Airflow is the de facto standard for data pipeline orchestration.
You describe pipelines as DAGs in Python code — you get version control, testing, and code reuse.
Two rules prevent 90% of Airflow disasters: <code>catchup=False</code> and idempotent tasks.
</div>

### The Citi Context
Citi's APM data pipeline ran nightly across 6,000 servers. Before Airflow:
shell scripts, cron jobs, no retry logic, no monitoring, silent failures.
After Airflow: DAG with retries, XCom for row count validation, email alerting,
and idempotent S3 writes. Mean-time-to-detect pipeline failures dropped from 6 hours to 90 seconds.

### Core Concepts

```
DAG (Directed Acyclic Graph)
├── Task 1: Extract metrics (PythonOperator)
│   └── Task 2: Validate row count (PythonOperator)   ← depends on Task 1
│       ├── Task 3a: Load to S3 (S3Hook)              ← depends on Task 2
│       └── Task 3b: Update catalog (GlueCrawler)     ← depends on Task 2
│           └── Task 4: Send alert (EmailOperator)    ← depends on Task 3b (on failure)
```

| Component | Role |
|-----------|------|
| **DAG** | Workflow definition — tasks + dependencies |
| **Task** | Unit of work — implemented as an Operator |
| **Operator** | Task type: PythonOperator, BashOperator, S3Hook, GlueCrawlerOperator |
| **Schedule** | Cron expression: `@daily` = `0 0 * * *` |
| **XCom** | Cross-task message passing (small values only — not DataFrames) |
| **Sensor** | Waits for external condition (S3FileExists, ExternalTask) |


In [ ]:
# ── A production-grade Airflow DAG ───────────────────────────────────────────
# Note: This code is illustrative — run it in an Airflow environment
# In this notebook, we show the structure and explain each decision

DAG_CODE = """
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.providers.amazon.aws.operators.glue_crawler import GlueCrawlerOperator

# Default arguments inherited by ALL tasks in this DAG
default_args = {
    "owner":            "data-engineering",
    "retries":          3,
    "retry_delay":      timedelta(minutes=5),
    "email_on_failure": True,
    "email":            ["de-alerts@citi.com"],
    "depends_on_past":  False,     # don't wait for previous run to succeed
}

with DAG(
    dag_id          = "daily_server_metrics",
    schedule_interval = "0 6 * * *",        # 6 AM UTC daily
    start_date      = datetime(2026, 1, 1),
    catchup         = False,                 # ← CRITICAL: no backfilling on first deploy
    default_args    = default_args,
    tags            = ["capacity", "telemetry"],
    max_active_runs = 1,                     # prevent concurrent runs
) as dag:

    def extract_metrics(**context):
        ds = context["ds"]   # execution date: "2026-02-27"
        # Pull from APM API, write raw JSON to s3://citi-raw/apm/{ds}/
        # ...
        rows = 6124   # simulate: fetched 6,124 rows
        # Push to XCom so downstream tasks can validate
        context["task_instance"].xcom_push(key="row_count", value=rows)
        return {"rows": rows, "date": ds}

    def validate_metrics(**context):
        ti = context["task_instance"]
        upstream = ti.xcom_pull(task_ids="extract_metrics", key="row_count")
        if upstream < 5000:
            raise ValueError(f"Row count too low: {upstream} (expected >= 5000)")
        print(f"Validation passed: {upstream} rows")

    def transform_and_load(**context):
        ds = context["ds"]
        # Read from raw S3, transform, write Parquet to s3://citi-processed/apm/{ds}/
        # This is idempotent: overwrite the partition if it already exists
        pass

    extract  = PythonOperator(task_id="extract_metrics",   python_callable=extract_metrics)
    validate = PythonOperator(task_id="validate_metrics",  python_callable=validate_metrics)
    load     = PythonOperator(task_id="transform_and_load", python_callable=transform_and_load)
    crawl    = GlueCrawlerOperator(task_id="update_catalog",
                                   config={"Name": "server-metrics-crawler"})

    # Define dependencies with >> (bitshift)
    extract >> validate >> load >> crawl
"""

print(DAG_CODE)
print("\n--- Key architectural decisions in this DAG: ---")
print("1. catchup=False: no backfilling on first deploy")
print("2. max_active_runs=1: prevents concurrent runs on same partition")
print("3. XCom for row count: validate before loading (fail fast)")
print("4. Idempotent load: overwrite S3 partition → safe to retry")
print("5. retries=3 with 5-min delay: handles transient API failures")


In [ ]:
# ── XCom: cross-task communication ───────────────────────────────────────────
# XCom = cross-communication. Tasks push small values, downstream tasks pull them.
# CRITICAL LIMITATION: XCom stores values in the Airflow metadata DB.
# Never push DataFrames, large lists, or file content via XCom.
# Push: row counts, file paths, status flags, small dicts.

XCOM_EXAMPLE = """
# Task 1: extract — pushes row count
def extract(**context):
    ti = context["task_instance"]
    rows = 6124
    ti.xcom_push(key="row_count", value=rows)
    ti.xcom_push(key="s3_path", value="s3://citi-raw/apm/2026-02-27/part-00001.json")

# Task 2: validate — pulls row count from Task 1
def validate(**context):
    ti = context["task_instance"]
    row_count = ti.xcom_pull(task_ids="extract", key="row_count")
    s3_path   = ti.xcom_pull(task_ids="extract", key="s3_path")
    assert row_count >= 5000, f"Too few rows: {row_count}"

# Task 3: load — uses S3 path from Task 1
def load(**context):
    ti = context["task_instance"]
    s3_path = ti.xcom_pull(task_ids="extract", key="s3_path")
    # read from s3_path, transform, write to processed/
"""

print(XCOM_EXAMPLE)
print("XCom rules:")
print("  ✅ Push: row_count (int), s3_path (str), status (bool), config (small dict)")
print("  ❌ Never push: DataFrames, Spark DataFrames, file contents, large lists")
print("  Why: XCom is stored in the Airflow metadata DB (PostgreSQL).")
print("  Large values = slow serialization + metadata DB bloat")


In [ ]:
# ── Idempotency: the most important Airflow principle ────────────────────────
# A DAG run is idempotent if re-running it produces the SAME result, not duplicate data.
# Required because: Airflow retries failed tasks automatically.
# If task fails after partially writing data, retry must produce clean output.

print("=== IDEMPOTENCY PATTERNS ===")
print()

# Pattern 1: S3 partition overwrite (Spark / PySpark)
SPARK_IDEMPOTENT = """
# ❌ Non-idempotent: appends on retry → duplicate data
df.write.mode("append").parquet("s3://bucket/date=2026-02-27/")

# ✅ Idempotent: overwrites partition on retry → same output
df.write.mode("overwrite").partitionBy("date").parquet("s3://bucket/")
"""
print("Pattern 1: Spark partition overwrite")
print(SPARK_IDEMPOTENT)

# Pattern 2: SQL DELETE then INSERT
SQL_IDEMPOTENT = """
-- ❌ Non-idempotent: INSERT on retry → duplicate rows
INSERT INTO daily_summary SELECT * FROM staging WHERE report_date = '2026-02-27';

-- ✅ Idempotent: DELETE then INSERT → same output on retry
DELETE FROM daily_summary WHERE report_date = '2026-02-27';
INSERT INTO daily_summary SELECT * FROM staging WHERE report_date = '2026-02-27';
"""
print("Pattern 2: DELETE + INSERT")
print(SQL_IDEMPOTENT)

# Pattern 3: ON CONFLICT (upsert)
SQL_UPSERT = """
-- ✅ Idempotent UPSERT: insert if new, update if exists
INSERT INTO daily_summary (server_id, report_date, avg_cpu)
SELECT server_id, report_date, avg_cpu FROM staging
ON CONFLICT (server_id, report_date) DO UPDATE SET avg_cpu = EXCLUDED.avg_cpu;
"""
print("Pattern 3: UPSERT (ON CONFLICT)")
print(SQL_UPSERT)


## 🎤 5 Interview Q&A

**Q1: What is `catchup=False` and why do you almost always want it?**

Without `catchup=False`, Airflow backfills all DAG runs from `start_date` to today.
If you deploy a DAG with `start_date=2026-01-01` today (March), Airflow queues
~60 historical runs immediately — flooding your workers and potentially inserting
60 copies of data. `catchup=False` prevents this: only future scheduled runs execute.
Set `catchup=True` only when you explicitly want historical backfilling (e.g., loading
historical data for the first time). Otherwise: always `catchup=False`.

---

**Q2: What is an XCom and what is its most important limitation?**

XCom (cross-communication) passes small values between tasks via the Airflow metadata DB.
One task pushes: `ti.xcom_push(key="row_count", value=6124)`.
Another task pulls: `ti.xcom_pull(task_ids="extract", key="row_count")`.
**Critical limitation:** XCom is stored in the metadata DB (PostgreSQL/MySQL).
Never push DataFrames, file contents, or large lists — they serialize slowly and bloat the DB.
XCom is for metadata: row counts, S3 file paths, status flags, small config dicts.

---

**Q3: What happens when an Airflow task fails?**

It retries `retries` times (from `default_args`) with `retry_delay` between attempts.
After exhausting retries: task status = FAILED. Downstream tasks that depend on it
are marked `UPSTREAM_FAILED` and don't execute. Airflow sends email if `email_on_failure=True`.
The DAG run is marked FAILED. You can manually clear the failed task to retry it after fixing the root cause.

---

**Q4: How do you make an Airflow DAG idempotent?**

Idempotent = running the same DAG run multiple times produces the same result (no duplicates).
Three patterns: (1) Spark: `mode("overwrite")` on the date partition — retry overwrites, no duplicates.
(2) SQL: `DELETE FROM table WHERE date = ds; INSERT...` — delete before inserting.
(3) UPSERT: `INSERT ... ON CONFLICT DO UPDATE` — merge semantics.
The execution date `{{ ds }}` scopes every read and write — each run processes exactly its own date.

---

**Q5: What is the difference between `LocalExecutor`, `CeleryExecutor`, and `KubernetesExecutor`?**

`LocalExecutor`: runs tasks as subprocesses on the Airflow scheduler machine. Simple, no extra infra.
Good for: small teams, < 20 DAGs, single-machine setups.
`CeleryExecutor`: distributes tasks to a pool of worker machines via a message broker (Redis/RabbitMQ).
Good for: medium scale, predictable workloads, existing team Celery expertise.
`KubernetesExecutor`: runs each task in its own Kubernetes pod. Scales to zero when idle.
Good for: variable workloads, per-task resource customization, cloud-native environments.
At Citi scale: CeleryExecutor with 5 workers was sufficient for ~200 DAGs.


## 📚 Key Terms

| Term | Definition |
|------|------------|
| **DAG** | Directed Acyclic Graph — the workflow definition. Contains tasks + dependencies. |
| **Task** | Unit of work inside a DAG — implemented as an Operator |
| **Operator** | Task type: PythonOperator, BashOperator, SparkSubmitOperator, etc. |
| **Schedule Interval** | Cron expression controlling when the DAG runs |
| **catchup=False** | Prevents backfilling missed DAG runs since start_date |
| **XCom** | Cross-task message passing — stored in metadata DB, small values only |
| **Sensor** | Task that waits for an external condition before proceeding |
| **Executor** | How Airflow runs tasks: Local, Celery, or Kubernetes |
| **Idempotency** | Re-running produces the same result — critical for safe retries |
| **Execution Date** | The logical date of a DAG run — `{{ ds }}` in Jinja templates |
| **Connection** | Stored credential (S3, Glue, Redshift) — reusable across DAGs |
| **Variable** | Key-value config store — accessible in any DAG |

## 🎯 Summary

### The Two Rules That Prevent 90% of Airflow Disasters
1. `catchup=False` — no unintended backfilling on deploy
2. Idempotent tasks — safe to retry without duplicate data

### Dependency Syntax
```python
extract >> validate >> load >> crawl    # linear
extract >> [validate_a, validate_b]     # fan-out
[validate_a, validate_b] >> load        # fan-in
```

### Interview Confidence Checklist
- [ ] Can explain what a DAG is and how tasks relate
- [ ] Can explain catchup=False and why it matters
- [ ] Can explain XCom limitation (never DataFrames)
- [ ] Can explain three idempotency patterns
- [ ] Can name the Citi narrative: 6h MTTD → 90s

**"Simplicity and clarity is Gold."** — Sean's Study Mantra 🚀
